In [ ]:
# Custom
import sys
sys.path.insert(0, '../src')
from args import args
import utils
from snippet_creation import SnippetCreation


## Utils and Inference
import csv
from collections import Counter, OrderedDict
import json
import numpy as np
import pandas as pd
import re

import os
from tqdm import tqdm

In [ ]:
class args(args):

    setting = ["batching", "single_variable"][0]

    experimental_setting = ["vanilla", "full_source_code", "source_code_grep_chunk", "paragraph_code", "variable_k_lines"][-1]

    batch_method = ["record"][0]

    saving_path = f"../saved_data/inference/{setting}/{experimental_setting}_{batch_method}"

    variable_column = "VarName"
    program_path_column = "corrected_path"

    # Record-level batched data produced by Batch_Creation Record.ipynb
    data_path = '../data/batched_data/record/batched_data_record_25June2026_var_occ.csv'
    complete_data_path = "../data/static_analyzer_Downloaded_Data/all_application_get_all_variable_info_23_06_2026.json"

    k_lines = 10

    platform = ["llm_provider_1", "watsonx", "mse", "litellm", "azure"][3]

    batch_size = 15

In [ ]:
import sys
sys.path.insert(0, '..')
from prompts import icl_paragraph_code_prompts_batched

prompt = icl_paragraph_code_prompts_batched.icl_source_code_prompts_batched
print(prompt)

In [ ]:
## Check if the saving directories exist or not; If not, then create them
utils.create_save_directories()

In [ ]:
### Check if the experiments saving directories exist or not; If not, then create them
utils.ensure_exp_dirs_exist(args.saving_path)

## Data Preprocessing

In [ ]:
# Record-level batched data: one row per variable, with batch_variables list for the whole sibling group
data_df = pd.read_csv(args.data_path)
data_df.head()

In [ ]:
# Parse batch_variables from string representation back to list
data_df['batch_variables'] = data_df['batch_variables'].apply(eval)

In [ ]:
complete_data_df = pd.read_json(args.complete_data_path).rename(columns={"ProgramID": "ProgID"})


## Source Code Context Helpers

In [ ]:
def read_file_binary_cobol(full_path):
    try:
        with open(full_path, "rb") as f:
            content = f.read()
        return {
            "path": full_path,
            "content": content.decode("latin-1")
        }
    except Exception as e:
        return {
            "error": f"Found file at {full_path} but could not read it: {e}"
        }


def create_variable_definition_snippet(prog_id, app_name, var_name, var_id, file_name):
    """
    Create COBOL variable definition snippets from complete_data_df for a specific program and variable.
    Filters to include only:
    - The variable itself (matching var_name and var_id)
    - Variables that have the considered variable as Father or Ancestor
    """
    try:
        program_df = complete_data_df[
            (complete_data_df['ProgID'] == prog_id) &
            (complete_data_df['app_name'] == app_name)
        ]

        if program_df.empty:
            return ""

        if isinstance(var_id, (list, tuple)):
            var_id_list = var_id
        else:
            var_id_list = [var_id]

        varid_match = program_df['VarID'].isin(var_id_list)
        father_match = program_df['Father'].isin(var_id_list)
        ancestor_match = program_df['Ancestor'].apply(
            lambda x: any(str(vid) in str(x) for vid in var_id_list) if pd.notna(x) else False
        )

        filtered_df = program_df[varid_match | father_match | ancestor_match]

        if filtered_df.empty:
            return ""

        snippet_df = filtered_df[['VarID', 'VarName', 'PIC', 'szValues', 'iLevel', 'Father']].copy()
        snippet_df.columns = ['var_id', 'var_name', 'pic', 'sz_values', 'i_level', 'father']

        snippet_creator = SnippetCreation()
        snippets = snippet_creator.create_snippets(snippet_df.drop_duplicates())

        if snippets:
            snippet_content = "\n\n".join(snippets.values())
            return f"[start of snippet for {file_name}]\n{snippet_content}\n[end of snippet for {file_name}]"

        return ""

    except Exception as e:
        print(f"Warning: Could not create snippet for prog_id={prog_id}, app_name={app_name}, var_name={var_name}, var_id={var_id}: {e}")
        return ""

In [ ]:
def gather_context_for_batch(batch_variables, app_name, k_lines=None):
    """
    Build ONE shared source-code context for all variables in a batch.

    Strategy
    --------
    For each variable in the batch, iterate its variable-level occurrences
    (VarStartRow / VarEndRow) one by one.  For each occurrence expand a
    window of k_lines above and below and record every included line index
    in a set — this gives free deduplication across overlapping windows.
    The accumulated line sets are then converted to sorted, contiguous
    ranges that are extracted from the file.

    Snippet creation is done for ALL variables in the batch that belong
    to each program (not just the ones whose name was absent from the
    extracted lines) and the combined snippet block is prepended to the
    file_content_str.

    Parameters
    ----------
    batch_variables : list[dict]
        List of variable dicts from the batch_variables column.  Each dict
        must contain: VarName, VarID, ProgID, corrected_path, VarStartRow,
        VarEndRow  (all list-valued except VarName / VarID).
    app_name : str
        Application name used by create_variable_definition_snippet.
    k_lines : int | None
        Lines of context above and below each occurrence.  Defaults to
        args.k_lines when None.

    Returns
    -------
    list[str]  – one marked string per source file, suitable for merge_context()
    """
    if k_lines is None:
        k_lines = args.k_lines

    # -----------------------------------------------------------------
    # Step 1: For every (prog_id, file_path) pair collect the set of
    #         0-based line indices that must be included (k-line windows
    #         around every variable occurrence).  File size is unknown at
    #         this stage so we store raw (unbounded) ranges; clipping
    #         happens after the file is read.
    #   prog_key -> file_path
    #   prog_key -> set of (var_start_0idx, var_end_0idx) tuples (dedup)
    # -----------------------------------------------------------------
    prog_to_file: dict = OrderedDict()        # prog_id -> file_path
    prog_to_occurrences: dict = OrderedDict() # prog_id -> list[(start_0, end_0)]
    # Also track which variables belong to each prog_id for snippets
    prog_to_vars: dict = OrderedDict()        # prog_id -> list[var_dict]

    for var_dict in batch_variables:
        if not isinstance(var_dict, dict):
            continue

        prog_ids        = var_dict['ProgID']
        corrected_paths = var_dict[args.program_path_column]
        start_rows      = var_dict['VarStartRow']  # 1-based row numbers
        end_rows        = var_dict['VarEndRow']    # 1-based row numbers

        for prog_id, fpath, var_start, var_end in zip(
            prog_ids, corrected_paths, start_rows, end_rows
        ):
            if prog_id not in prog_to_file:
                prog_to_file[prog_id]        = fpath
                prog_to_occurrences[prog_id] = []
                prog_to_vars[prog_id]        = []

            # Convert 1-based row to 0-based index for list slicing
            entry = (int(var_start) - 1, int(var_end) - 1)
            if entry not in prog_to_occurrences[prog_id]:
                prog_to_occurrences[prog_id].append(entry)

        # Register the variable under every program it appears in (once)
        for prog_id in set(prog_ids):
            if prog_id in prog_to_vars and var_dict not in prog_to_vars[prog_id]:
                prog_to_vars[prog_id].append(var_dict)

    if not prog_to_file:
        return []

    # -----------------------------------------------------------------
    # Step 2: For each program, read the file once, build a deduplicated
    #         set of line indices, convert to contiguous ranges, and
    #         extract the code.
    # -----------------------------------------------------------------
    source_codes = []

    for prog_id in prog_to_file:
        file_path   = prog_to_file[prog_id]
        occurrences = prog_to_occurrences[prog_id]

        file_content = read_file_binary_cobol(full_path=file_path)
        if 'error' in file_content:
            continue

        file_name   = os.path.basename(file_path)
        source_code = file_content['content']
        lines       = source_code.splitlines()
        total_lines = len(lines)

        # Build k-line windows for every occurrence, clip to file bounds
        line_ranges = []
        for var_start_0, var_end_0 in occurrences:
            ctx_start = max(0, var_start_0 - k_lines)
            ctx_end   = min(total_lines - 1, var_end_0 + k_lines)
            line_ranges.append((ctx_start, ctx_end))

        # Sort and merge overlapping / adjacent ranges
        line_ranges.sort(key=lambda x: x[0])
        merged_ranges = []
        for start, end in line_ranges:
            if not merged_ranges:
                merged_ranges.append([start, end])
            else:
                last_start, last_end = merged_ranges[-1]
                if start <= last_end + 1:
                    merged_ranges[-1][1] = max(last_end, end)
                else:
                    merged_ranges.append([start, end])

        # Extract marked snippets for each merged range
        snippets = []
        for start, end in merged_ranges:
            if start < total_lines and end < total_lines:
                snippet = lines[start:end + 1]
                marked_snippet = (
                    f"[lines {start}-{end}]\n"
                    + "\n".join(snippet)
                    + f"\n[end lines {start}-{end}]"
                )
                snippets.append(marked_snippet)

        file_content_str = "\n".join(snippets)

        # -----------------------------------------------------------------
        # Step 3: Create COBOL definition snippets for ALL variables in the
        #         batch that belong to this program and prepend the combined
        #         snippet block to file_content_str.
        # -----------------------------------------------------------------
        all_snippet_parts = []
        for var_dict in prog_to_vars.get(prog_id, []):
            var_name = var_dict['VarName']
            var_id   = var_dict['VarID']
            snippet_str = create_variable_definition_snippet(
                prog_id, app_name, var_name, var_id, file_name
            )
            if snippet_str:
                all_snippet_parts.append(snippet_str)

        if all_snippet_parts:
            combined_snippets = "\n".join(all_snippet_parts)
            file_content_str = combined_snippets + "\n" + file_content_str

        marked_content = f"[start of {file_name}]\n{file_content_str}\n[end of {file_name}]"
        source_codes.append(marked_content)

    return source_codes


def merge_context(source_codes):
    return "\n\n".join(source_codes)

## Validating and Loading the Inference Pipeline

In [ ]:
inference = utils.Inference(args)

In [ ]:
model_names = utils.validate_api_endpoints(args, args.platform)

print(f"Models from {args.platform}: {model_names}\n")

model_names = ['llama3_3_70b', 'gpt_oss_120b', 'gpt_oss_20b', 'qwen_3_8B', 'llama4_17b_16e', 'qwen_3_30b_a3b_thinking', 'kimi_2_5', 'gemma_4_31B_it']

print(f"Selected Models from {args.platform}: {model_names}")

In [ ]:
pred_folder = args.saving_path + "/"
pred_folder

In [ ]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def write_json_record(file_path: str, record: dict) -> None:
    """Append a record to a JSON array file."""
    with open(file_path, 'r+') as jsonfile:
        data = json.load(jsonfile)
        data.append(record)
        jsonfile.seek(0)
        json.dump(data, jsonfile, indent=2, cls=NumpyEncoder)
        jsonfile.truncate()

## Inference — Record-Level Batching with Variable k-Lines Context

For every row in `data_df`:
1. Retrieve the batch of sibling variables (`batch_variables`).
2. Build **one** shared context: for each variable occurrence (`VarStartRow`/`VarEndRow`), expand a window of `k_lines` above and below; overlapping windows are merged and deduplicated.
3. COBOL definition snippets are created for **all** batch variables per program and prepended to the file context.
4. Fill the batched prompt template and run inference.

In [ ]:
file_names = []

for model_name in tqdm(model_names, desc="Models", position=0):
    prediction_df = data_df

    file_name = f"{model_name}_variable_DD_{args.experimental_setting}_{args.batch_method}_.json"
    file_name = utils.update_file_name(
        file_name, pred_folder=pred_folder, extension=".json"
    )

    # Checkpointing Logic — set model_name to '' and assign file_name / checkpoint_index manually
    if model_name == '':
        file_name = 'PLACEHOLDER_FILE_NAME.json'
        checkpoint_index = 0
    else:
        checkpoint_index = 0
        fields = list(prediction_df.columns) + ["prompt_used", "token_statistics", "predictions_DD"]

        with open(pred_folder + file_name, 'w') as jsonfile:
            json.dump([], jsonfile)

    print(f"Starting Inference for {model_name} in {file_name}")
    full_path = pred_folder + file_name

    pbar = tqdm(
        total=len(prediction_df),
        desc=f"Inference [{model_name}]",
        position=1,
        leave=False,
        dynamic_ncols=True
    )

    for i, row in prediction_df.iterrows():
        if i < checkpoint_index:
            pbar.update(1)
            continue
        if i == checkpoint_index:
            print(f"starting inference from index {i}")

        row_data = list(row)

        # -----------------------------------------------------------
        # Resolve batch variables and app context
        # -----------------------------------------------------------
        # batch_variables is a list of dicts (from Batch_Creation Record.ipynb)
        # each dict has 'VarName', 'VarID', 'ParaID', 'ProgramName', etc.
        batch_variables = row['batch_variables'][:args.batch_size]          # list[dict]
        app_name        = row['app_name']                 # str
        # Extract plain variable name strings for the prompt
        batch_var_name_strings = [
            d['VarName'] if isinstance(d, dict) else d
            for d in batch_variables
        ]

        # -----------------------------------------------------------
        # Build the shared context once for the whole batch:
        # iterate program IDs -> unique paragraphs per program
        # -----------------------------------------------------------
        try:
            source_codes = gather_context_for_batch(
                batch_variables=batch_variables,
                app_name=app_name
            )
            source_code_string = merge_context(source_codes)

        except Exception as e:
            response    = f"error found: {e}"
            token_stats = f"error found: {e}"
            record = dict(zip(fields, row_data + ["", token_stats, response]))
            write_json_record(full_path, record)
            pbar.update(1)
            continue

        # -----------------------------------------------------------
        # Fill the batched prompt template
        # -----------------------------------------------------------
        current_prompt = prompt[:]
        current_prompt = current_prompt.replace("{{SOURCE_CONTEXT}}", source_code_string)
        current_prompt = current_prompt.replace("{{BATCHED_INPUT}}", str(batch_var_name_strings))

        # -----------------------------------------------------------
        # Run inference
        # -----------------------------------------------------------
        try:
            response, raw_output = inference(
                prompt=current_prompt,
                inference_platform=args.platform,
                model_name=model_name,
                return_raw=True
            )

            token_stats = dict(dict(raw_output)['usage'])
            token_stats = {k: token_stats[k] for k in ('completion_tokens', 'prompt_tokens', 'total_tokens')}

        except Exception as e:
            response    = f"error found: {e}"
            token_stats = f"error found: {e}"

        record = dict(zip(fields, row_data + [current_prompt, token_stats, response]))
        write_json_record(full_path, record)
        pbar.update(1)

    #     break

    # break

    pbar.close()
    file_names.append(file_name)